In [7]:
import os
import glob
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

In [8]:
import os
import glob
import pandas as pd
import numpy as np
import json
import traceback # <-- Bổ sung thư viện để truy vết lỗi chi tiết
import warnings
warnings.filterwarnings('ignore')

print("[*] KHỞI ĐỘNG MODULE BÓC TÁCH DỮ LIỆU 52 TUẦN (PRICE & VOLUME)...")

def aggregate_52_weeks_dynamics(data_path='Data_Cube'):
    # Đọc danh sách 50 mã cổ phiếu mục tiêu
    try:
        df_micro = pd.read_csv("db_micro_nodes.csv")
        target_symbols = set(df_micro['Symbol'].unique())
    except FileNotFoundError:
        print("[!] Không tìm thấy db_micro_nodes.csv. Sẽ quét toàn bộ thư mục.")
        target_symbols = None

    # Quét toàn bộ file CSV trong thư mục data
    all_csvs = glob.glob(f"{data_path}/*/*/*.csv")
    list_yearly_stats = []

    for path in all_csvs:
        # Bỏ qua các file Báo cáo tài chính, chỉ lấy file giá
        if any(keyword in path for keyword in ["Finance", "Ratio", "Profile"]):
            continue
            
        parts = os.path.normpath(path).split(os.sep)
        if len(parts) < 3: continue
        
        symbol = parts[-2].upper() # Mã CK thường là tên thư mục cha
        if target_symbols and symbol not in target_symbols:
            continue

        try:
            df = pd.read_csv(path)
            
            # --- TỰ ĐỘNG NHẬN DIỆN CỘT DỮ LIỆU ---
            col_lower = [c.lower() for c in df.columns]
            
            # Tìm cột Ngày
            date_col = next((c for c, l in zip(df.columns, col_lower) if 'date' in l or 'ngay' in l or 'time' in l), None)
            
            # Ưu tiên tìm cột "Giá Điều Chỉnh" (Adj Close) trước!
            close_col = next((c for c, l in zip(df.columns, col_lower) if 'dieuchinh' in l or 'adj' in l), None)
            
            # Nếu bộ data gốc không có cột điều chỉnh, đành lấy Giá đóng cửa thô
            if not close_col:
                close_col = next((c for c, l in zip(df.columns, col_lower) if 'close' in l or 'dongcua' in l or 'gia' in l), None)
            
            # Tìm cột Khối Lượng
            vol_col = next((c for c, l in zip(df.columns, col_lower) if 'vol' in l or 'kl' in l or 'khoiluong' in l), None)
            
            if not date_col or not close_col or not vol_col:
                print(f"  [BỎ QUA] {symbol}: Không nhận diện đủ cột (Date: {date_col}, Close: {close_col}, Vol: {vol_col})")
                continue 
                
            # Xử lý chuỗi thời gian (Đã xóa đoạn lặp code)
            df[date_col] = pd.to_datetime(df[date_col])
            df = df.sort_values(date_col).set_index(date_col)
            
            # =================================================================
            # 🚀 THUẬT TOÁN TỰ ĐỘNG ĐIỀU CHỈNH CHIA TÁCH (SPLIT AUTO-CORRECTOR)
            # =================================================================
            daily_returns = df[close_col].pct_change()
            split_days = daily_returns < -0.16 

            if split_days.any():
                df['Adj_Factor'] = 1.0
                for date_idx in df[split_days].index:
                    loc = df.index.get_loc(date_idx)
                    if loc > 0:
                        prev_price = df.iloc[loc-1][close_col]
                        curr_price = df.iloc[loc][close_col]
                        split_ratio = curr_price / prev_price
                        
                        df.iloc[:loc, df.columns.get_loc('Adj_Factor')] *= split_ratio
                
                # Cập nhật đè Giá Thô thành Giá Điều Chỉnh
                df[close_col] = df[close_col] * df['Adj_Factor']
                print(f"    [*] Đã auto-fix lỗi chia tách cho {symbol} ({len(df[split_days])} lần chia)")
            # =================================================================

            df['Year'] = df.index.year
            years = df['Year'].unique()
            
            for y in years:
                df_y = df[df['Year'] == y]
                if len(df_y) < 50: 
                    continue
                    
                # RESAMPLE DỮ LIỆU VỀ CHU KỲ TUẦN (WEEKLY)
                weekly_df = df_y.resample('W').agg({
                    close_col: 'last',  
                    vol_col: 'sum'      
                }).dropna()
                
                if len(weekly_df) < 10: continue
                
                weekly_close = weekly_df[close_col]
                weekly_vol = weekly_df[vol_col]
                
                # 1. Đóng gói JSON
                close_list = json.dumps([round(x, 2) for x in weekly_close.tolist()])
                
                # 2. Max Drawdown
                rolling_max = weekly_close.cummax()
                drawdown = (weekly_close - rolling_max) / rolling_max
                max_drawdown = drawdown.min() * 100 
                
                # 3. Volatility
                weekly_returns_stat = weekly_close.pct_change().dropna()
                volatility = weekly_returns_stat.std() * np.sqrt(52) * 100 
                
                # 4. Volume Spikes
                mean_vol = weekly_vol.mean()
                spike_weeks = len(weekly_vol[weekly_vol > 2 * mean_vol]) 
                
                list_yearly_stats.append({
                    'Symbol': symbol,
                    'Year': y,
                    'Weekly_Close_List': close_list,
                    'Max_Drawdown_%': round(max_drawdown, 2),
                    'Volatility_%': round(volatility, 2),
                    'Volume_Spike_Weeks': spike_weeks,
                    'Avg_Weekly_Volume': round(mean_vol, 0)
                })
                
            print(f"  [+] Đã bóc tách mượt mà 52 tuần cho mã: {symbol}")
            
        except Exception as e:
            # IN CHI TIẾT DÒNG LỖI ĐỂ TÌM BUG
            print(f"  [LỖI] Xử lý mã {symbol} thất bại!")
            print(traceback.format_exc())
            print("-" * 50)
            
    # LƯU DATABASE
    if list_yearly_stats:
        master_dynamics = pd.DataFrame(list_yearly_stats)
        master_dynamics.to_csv("db_market_dynamics.csv", index=False)
        print(f"\n[+] HOÀN TẤT XUẤT SẮC! Đã lưu thành 'db_market_dynamics.csv' (Kích thước: {master_dynamics.shape})")
    else:
        print("\n[!] Không gom được dữ liệu. Hãy kiểm tra lại file giá trong Data_Cube.")

aggregate_52_weeks_dynamics()

[*] KHỞI ĐỘNG MODULE BÓC TÁCH DỮ LIỆU 52 TUẦN (PRICE & VOLUME)...
    [*] Đã auto-fix lỗi chia tách cho DIG (4 lần chia)
  [+] Đã bóc tách mượt mà 52 tuần cho mã: DIG
    [*] Đã auto-fix lỗi chia tách cho VHM (2 lần chia)
  [+] Đã bóc tách mượt mà 52 tuần cho mã: VHM
    [*] Đã auto-fix lỗi chia tách cho VIC (7 lần chia)
  [+] Đã bóc tách mượt mà 52 tuần cho mã: VIC
  [+] Đã bóc tách mượt mà 52 tuần cho mã: VRE
    [*] Đã auto-fix lỗi chia tách cho KDH (2 lần chia)
  [+] Đã bóc tách mượt mà 52 tuần cho mã: KDH
    [*] Đã auto-fix lỗi chia tách cho CEO (1 lần chia)
  [+] Đã bóc tách mượt mà 52 tuần cho mã: CEO
    [*] Đã auto-fix lỗi chia tách cho PDR (2 lần chia)
  [+] Đã bóc tách mượt mà 52 tuần cho mã: PDR
    [*] Đã auto-fix lỗi chia tách cho NVL (3 lần chia)
  [+] Đã bóc tách mượt mà 52 tuần cho mã: NVL
    [*] Đã auto-fix lỗi chia tách cho KBC (3 lần chia)
  [+] Đã bóc tách mượt mà 52 tuần cho mã: KBC
    [*] Đã auto-fix lỗi chia tách cho VCG (1 lần chia)
  [+] Đã bóc tách mượt mà